# 🚀 GGUF Vision Studio - Dual T4 GPU (Kaggle) + Cloudflare Tunnel
Este notebook roda modelos **GGUF Multimodais (Visão)** com projetor `mmproj-F16.gguf` no **Kaggle** com **2x NVIDIA T4 (32GB VRAM total)** dividido via `--tensor-split 1,1` e expõe uma API pública segura via **Cloudflare Tunnel** compatível com KoboldCPP / OpenAI / SillyTavern e a interface Web GGUF Vision Studio.

> **Configurações Ativas:**
- **Modelo Base:** `gemma-4-E4B-it-UD-Q4_K_XL.gguf`
- **Projetor Visão:** `mmproj-F16.gguf`
- **Tensor Split (Dual T4):** `1,1` (50% GPU 0 / 50% GPU 1)
- **Contexto (n_ctx):** `8192` tokens
- **GPU Offload:** `-ngl 99` (100% das camadas aceleradas nas GPUs)

> **Atenção:** No menu superior do Kaggle, certifique-se de configurar **Accelerator: GPU T4 x2** e **Internet: ON**!

In [ ]:
# 1. Verificar GPUs T4 disponíveis
!nvidia-smi

import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
print(f"Total GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} - {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Instalar utilitários e Cloudflared Tunnel
import os, urllib.request, subprocess, time

print("⏳ Baixando Cloudflared Tunnel...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

print("⏳ Instalando utilitários e dependências...")
!apt-get update -qq && apt-get install -y -qq aria2 psmisc libgomp1
!pip install -q huggingface_hub requests

In [ ]:
# 3. Baixar binário otimizado ou compilar llama-server com CUDA (sm_75 para Dual T4)
import os
if not os.path.exists("llama-server") and not os.path.exists("llama.cpp/build/bin/llama-server"):
    print("⏳ Obtendo release otimizada do llama.cpp para CUDA...")
    !wget -q -nc https://github.com/ggerganov/llama.cpp/releases/download/b4770/llama-b4770-bin-ubuntu-x64.zip -O llama-bin.zip || true
    if os.path.exists("llama-bin.zip") and os.path.getsize("llama-bin.zip") > 1000000:
        !unzip -q -o llama-bin.zip -d llama_bin && cp llama_bin/llama-server ./llama-server && chmod +x ./llama-server
    else:
        print("Compilando llama.cpp com CUDA (sm_75 para Dual T4)...")
        !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
        !cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="75"
        !cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-server
        !cp llama.cpp/build/bin/llama-server ./llama-server

!chmod +x ./llama-server
print("✅ llama-server pronto!")

In [ ]:
# 4. Baixar modelo GGUF e projetor multimodal (mmproj-F16.gguf) com aria2c
import os
os.makedirs("models", exist_ok=True)

MODEL_URL = "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/gemma-4-E4B-it-UD-Q4_K_XL.gguf"
MMPROJ_URL = "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-F16.gguf"
MODEL_PATH = "models/gemma-4-E4B-it-UD-Q4_K_XL.gguf"
MMPROJ_PATH = "models/mmproj-F16.gguf"

if not os.path.exists(MODEL_PATH):
    print(f"📥 Baixando modelo principal: {MODEL_PATH} ...")
    !aria2c -x 16 -s 16 -k 1M -c "{MODEL_URL}" -d models -o "gemma-4-E4B-it-UD-Q4_K_XL.gguf"
else:
    print(f"✅ Modelo já existente: {MODEL_PATH}")

if not os.path.exists(MMPROJ_PATH):
    print(f"📥 Baixando projetor multimodal mmproj: {MMPROJ_PATH} ...")
    !aria2c -x 16 -s 16 -k 1M -c "{MMPROJ_URL}" -d models -o "mmproj-F16.gguf"
else:
    print(f"✅ mmproj já existente: {MMPROJ_PATH}")

!ls -lh models/

In [ ]:
# 5. Iniciar Servidor llama.cpp com Dual T4 Split + Cloudflare Tunnel
import subprocess, time, re, sys, os

# Matar processos anteriores se houver
!fuser -k 8080/tcp 2>/dev/null || true
!killall cloudflared 2>/dev/null || true

# Comando para rodar o llama-server com divisão nas duas GPUs T4 (CUDA 0 e 1)
server_cmd = [
    "./llama-server",
    "-m", "models/gemma-4-E4B-it-UD-Q4_K_XL.gguf",
    "--mmproj", "models/mmproj-F16.gguf",
    "-ngl", "99",
    "--split-mode", "row",
    "--tensor-split", "1,1",
    "-c", "8192",
    "-fa",
    "--host", "0.0.0.0",
    "--port", "8080",
    "--parallel", "1",
    "-np", "1"
]

print("🚀 Iniciando llama-server em segundo plano...")
llama_proc = subprocess.Popen(server_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(5)

print("🌐 Iniciando Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8080"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

cloudflare_url = None
start_time = time.time()
while time.time() - start_time < 35:
    line = tunnel_proc.stdout.readline()
    if line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            cloudflare_url = match.group(0)
            break
    time.sleep(0.1)

if cloudflare_url:
    print("\n" + "="*70)
    print(f"🎉 SEU TÚNEL CLOUDFLARE ESTÁ ONLINE!")
    print(f"🔗 URL Pública da API: {cloudflare_url}")
    print(f"🔗 Endpoint OpenAI:    {cloudflare_url}/v1/chat/completions")
    print(f"🔗 Endpoint Kobold:    {cloudflare_url}/completion")
    print(f"🔗 Endpoint Modelos:   {cloudflare_url}/v1/models")
    print("="*70 + "\n")
    print("Copie a URL acima e cole na interface web GGUF Vision Studio ou no SillyTavern!")
else:
    print("⚠️ Não foi possível capturar a URL do Cloudflare automaticamente. Verifique os logs abaixo.")

try:
    while True:
        out = llama_proc.stdout.readline()
        if out:
            print(out.strip())
        time.sleep(0.1)
except KeyboardInterrupt:
    print("\n🛑 Encerrando servidor e túnel...")
    llama_proc.terminate()
    tunnel_proc.terminate()

In [ ]:
# 6. Teste de Visão (Opcional - execute para testar inferência de imagem com a API)
import requests, base64

!wget -q -nc https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/420px-PNG_transparency_demonstration_1.png -O test.png

with open("test.png", "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("utf-8")

payload = {
    "model": "gemma-4-E4B-it",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Descreva o que você vê nesta imagem em português."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
            ]
        }
    ],
    "max_tokens": 300,
    "temperature": 0.2
}

res = requests.post("http://127.0.0.1:8080/v1/chat/completions", json=payload)
print("Status:", res.status_code)
if res.status_code == 200:
    print("Resposta do Modelo:")
    print(res.json()["choices"][0]["message"]["content"])
else:
    print(res.text)